In [3]:
!pip install gensim

In [4]:
import nltk
import gensim
import numpy as np
import pandas as pd
import string
import matplotlib.pyplot as plt
from nltk.corpus import brown
from nltk.tokenize import word_tokenize, sent_tokenize
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
nltk.download('brown')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

# **Task-1**

In [6]:
df_full = pd.read_csv('/content/drive/MyDrive/train.csv')
df_full.columns = ['label', 'title', 'content']
print(f"Full train set shape: {df_full.shape}")

Full train set shape: (3599999, 3)


In [7]:
df = df_full.tail(50000).copy()                              #.head(), .iloc[start:end], .sample(n=50000, random_state=42)

df = df.drop(columns=['title'])
print(f"Shape after dropping title: {df.shape}")
display(df.head())

df['content'] = df['content'].str.lower().str.replace(f'[{string.punctuation}]', '', regex=True)
display(df.head())

bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(df['content'])

display(X_bow.shape)

Shape after dropping title: (50000, 2)


,label,content
3549999,2,One of the best roots music albums I've ever h...
3550000,2,"Being a longtime fan of the blues and John's,i..."
3550001,2,Uno attack is fun for all ages. Glad we got it...
3550002,2,This is a game the whole family can enjoy. And...
3550003,2,I brought the cards for my friend. The cards t...


,label,content
3549999,2,one of the best roots music albums ive ever he...
3550000,2,being a longtime fan of the blues and johnsit ...
3550001,2,uno attack is fun for all ages glad we got it ...
3550002,2,this is a game the whole family can enjoy and ...
3550003,2,i brought the cards for my friend the cards th...


(50000, 109265)

# **Task-2**

In [8]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['content'])

feature_names = tfidf_vectorizer.get_feature_names_out()     #return an array of all feature
mean_tfidf_scores = X_tfidf.mean(axis=0).A1                  #np.array(X_tfidf.mean(axis=0)).flatten() # return an array of mean of feature

tfidf_df = pd.DataFrame({'word': feature_names, 'mean_tfidf': mean_tfidf_scores})  #creat a DataFrame
top25 = tfidf_df.nlargest(25, 'mean_tfidf').reset_index()                          #.reset_index(level=None, drop=False, inplace=False, col_level=0, col_fill='')

display(top25)

,index,word,mean_tfidf
0,95859,the,0.088173
1,7160,and,0.053680
2,97725,to,0.051907
3,51114,it,0.051307
4,67086,of,0.042058
5,96624,this,0.040902
6,50876,is,0.040792
7,38382,for,0.031424
8,48572,in,0.029810
9,104597,was,0.028580


# **Task-3**

In [9]:
glove_file = '/content/drive/MyDrive/glove.6B.100d.txt'

In [10]:
def load_glove_model(glove_file): #converts the text file into dictionary{key(word):value(vector)}
    word_vectors = {}
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.array(values[1:], dtype='float32')
            word_vectors[word] = vector
    return word_vectors
glove_model = load_glove_model(glove_file)

In [11]:
paris_vector     = glove_model['paris']
france_vector    = glove_model['france']
italy_vector     = glove_model['italy']
rome_vector      = glove_model['rome']
resulting_vector = paris_vector - france_vector + italy_vector
#display(resulting_vector)

result_rome = cosine_similarity([resulting_vector], [rome_vector])
print(result_rome)
print(f"Cosine Similarity between 'resulting_vector' and 'rome_vector': {result_rome[0][0]}")

[[0.8084001]]
Cosine Similarity between 'resulting_vector' and 'rome_vector': 0.8084000945091248


# **Task-4**

In [12]:
sentences  = [[token.lower() for token in sent] for sent in brown.sents()]

model_sg = Word2Vec(vector_size=10, window=3, sg=1, min_count=1)          #defines the model architecture
voc_sg = model_sg.build_vocab(sentences)                                  #makes a dictionary{unique word : random vector}
model_sg.train(sentences, total_examples=len(sentences), epochs=10)       #trains the model

model_cbow = Word2Vec(vector_size=10, window=3, sg=0, min_count=1)
voc_cbow = model_cbow.build_vocab(sentences)
model_cbow.train(sentences, total_examples=len(sentences), epochs=10)

(8362513, 11611920)

In [13]:
#shoter version
#sentences  = [[token.lower() for token in sent] for sent in brown.sents()]
#model_sg   = Word2Vec(sentences, vector_size=10, window=3, sg=1, min_count=1, epochs=10)
#model_cbow = Word2Vec(sentences, vector_size=10, window=3, sg=0, min_count=1, epochs=10)

In [14]:
def same_analogy(model, paris, france, italy):
  result = model.wv.most_similar(positive=[paris, italy], negative=[france], topn=3)
  return result

sq_evaluate   = same_analogy(model_sg, 'paris', 'france', 'italy')
cbow_evaluate = same_analogy(model_cbow, 'paris', 'france', 'italy')

display(sq_evaluate)
display(cbow_evaluate)

[('winslow', 0.9701487421989441),
 ('yesterday', 0.9633877873420715),
 ('afternoon', 0.962253212928772)]

[('drawn', 0.9939234256744385),
 ('dancing', 0.9927611947059631),
 ('thrown', 0.9922003746032715)]

In [15]:
# def suitable_analogy(model, paris, france, italy):
#   resulting_vector = model.wv['paris'] - model.wv['france'] + model.wv['italy']
#   display(resulting_vector)

# suitable_analogy(model_sg, 'paris', 'france', 'italy')
# suitable_analogy(model_cbow, 'paris', 'france', 'italy')